# Feature Circuits Phase B: SAE on MoE Sub-Block Output

**Hypothesis** (from Phase A null result): circuits may not live in the *residual stream* but in the per-sub-block computation (MoE outputs). Residual is a summary bus; MoE expert routing is where token-specific compute happens.

**Test**: capture the MoE sub-block's OUTPUT (delta it adds to residual) at L11, L17, L23. Train SAEs on those. Check if:
1. MoE features show stronger cross-layer fire↔pre correlation than residual features
2. MoE features at same layer correlate differently with residual features (sanity check)

**Budget**: ~1.5h on RTX 6000. Forward 200 Stage B prompts + train 3 SAEs + alignment analysis.

**Output**: `caiovicentino1/qwen36-feature-circuits/phase_b/` with MoE SAEs + correlations.

## Cell 1 — Setup

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path

def pip(*a): return subprocess.run([sys.executable, '-m', 'pip', *a], check=False)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception:
    ok = False

if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'hf_transfer')
    pip('uninstall', '-y', 'transformers', 'causal-conv1d')
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC)
    pip('install','--no-cache-dir','flash-linear-attention')
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'):
            del sys.modules[m]

try:
    from google.colab import userdata
    t = userdata.get('HF_TOKEN')
    if t:
        from huggingface_hub import login
        login(token=t); print('HF auth OK')
except: pass

import json, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from safetensors import safe_open
from huggingface_hub import snapshot_download, HfApi, create_repo, login
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/fc_phase_b'); OUT.mkdir(exist_ok=True)
HF_OUT = 'caiovicentino1/qwen36-feature-circuits'
HF_STAGE_B = 'caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b'
MODEL_ID = 'Qwen/Qwen3.6-35B-A3B'

CAPTURE_LAYERS = [11, 17, 23]
D_MODEL = 2048
N_FEATURES = 4096
TOPK = 32
SAE_EPOCHS = 25
SAE_BS = 4096
SAE_LR = 3e-4
MAX_PROMPT_LEN = 2048
N_PROMPTS = 200
MAX_TOKENS_PER_LAYER = 300_000

print('setup done')

## Cell 2 — Load Qwen3.6

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
for p in model.parameters(): p.requires_grad_(False)
print(f'Model: {torch.cuda.memory_allocated()/1e9:.1f} GB')

def get_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers')]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except: continue
    raise RuntimeError(f'layer {idx} not found')

# Inspect MoE block structure
sample_layer = get_layer_module(model, CAPTURE_LAYERS[0])
print(f'\nLayer {CAPTURE_LAYERS[0]} top-level children:')
for n, m in sample_layer.named_children():
    print(f'  {n}: {type(m).__name__}')

# Find the MoE sub-block (the part that returns what gets added to residual)
# In Qwen3_5MoE: layer has .mlp which is the MoE module

## Cell 3 — Hook MoE sub-block outputs

Hook `layer.mlp`'s forward output — that's the delta MoE adds to residual.

In [ ]:
# Find the MoE module within the layer
def find_moe_module(layer):
    for n, m in layer.named_children():
        if n == 'mlp':
            return m, n
    return None, None

moe_modules = {}
for L in CAPTURE_LAYERS:
    layer = get_layer_module(model, L)
    mod, name = find_moe_module(layer)
    moe_modules[L] = mod
    print(f'L{L}: {name} type={type(mod).__name__}')

# Capture MoE sub-block output (tuple returned -> take first element)
captured_moe = {L: None for L in CAPTURE_LAYERS}

def make_hook(L):
    def hook(module, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        captured_moe[L] = h.detach()
    return hook

handles = []
for L in CAPTURE_LAYERS:
    h = moe_modules[L].register_forward_hook(make_hook(L))
    handles.append(h)
print(f'\nOK {len(handles)} MoE hooks installed')

## Cell 4 — Load Stage B questions + format prompts

In [ ]:
corpus = snapshot_download(HF_STAGE_B, repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))

questions = []
seen = set()
for shard in shards:
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        q = meta['question']
        if q in seen: continue
        seen.add(q)
        opts = json.loads(meta['options'])
        if len(opts) != 10: continue
        questions.append(dict(question=q, options=opts, gold_letter=meta['gold_letter']))

random.seed(42); random.shuffle(questions)
questions = questions[:N_PROMPTS]
print(f'{len(questions)} questions loaded')

def format_prompt(q, opts):
    choices = '\n'.join(f'{chr(65+i)}. {o}' for i, o in enumerate(opts))
    content = ("Answer the following multiple-choice question step by step.\n\n"
        f"Question: {q}\n\nOptions:\n{choices}\n\nAnswer:")
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True,
                                    enable_thinking=False)

## Cell 5 — Forward prompts + capture MoE outputs per token

Only forward the prompt (no generation). Keeps it fast + consistent with Stage B capture semantics.

In [ ]:
moe_acts = {L: [] for L in CAPTURE_LAYERS}
total_tokens = {L: 0 for L in CAPTURE_LAYERS}
t0 = time.time()

for q in tqdm(questions, desc='forward'):
    try:
        p = format_prompt(q['question'], q['options'])
        ids = tok(p, return_tensors='pt').input_ids.cuda()
        if ids.shape[1] > MAX_PROMPT_LEN: continue
        with torch.no_grad():
            _ = model(input_ids=ids, attention_mask=torch.ones_like(ids))
        for L in CAPTURE_LAYERS:
            if captured_moe[L] is None: continue
            a = captured_moe[L][0].float().cpu()  # [T, d_model]
            moe_acts[L].append(a)
            total_tokens[L] += a.shape[0]
        # Cap to prevent RAM blowup
        if any(total_tokens[L] >= MAX_TOKENS_PER_LAYER for L in CAPTURE_LAYERS):
            break
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); continue
    except Exception as e:
        continue

for h in handles: h.remove()
elapsed = (time.time()-t0)/60
print(f'\nForward done in {elapsed:.1f} min')

# Stack per layer
acts_moe = {}
for L in CAPTURE_LAYERS:
    X = torch.cat(moe_acts[L], dim=0)
    if X.shape[0] > MAX_TOKENS_PER_LAYER:
        idx = torch.randperm(X.shape[0])[:MAX_TOKENS_PER_LAYER]
        X = X[idx]
    acts_moe[L] = X
    print(f'L{L} MoE: {tuple(X.shape)} dtype={X.dtype} mean_norm={X.norm(dim=-1).mean().item():.3f}')

# Release hook storage
for L in CAPTURE_LAYERS: captured_moe[L] = None
moe_acts = None
torch.cuda.empty_cache()

## Cell 6 — Train TopK SAEs on MoE outputs (same recipe as Phase A)

In [ ]:
class TopKSAE(nn.Module):
    def __init__(self, d_in, n, k):
        super().__init__()
        self.W_enc = nn.Parameter(torch.empty(d_in, n))
        self.b_enc = nn.Parameter(torch.zeros(n))
        self.W_dec = nn.Parameter(torch.empty(n, d_in))
        self.b_dec = nn.Parameter(torch.zeros(d_in))
        self.k = k
        w = torch.randn(d_in, n) / (d_in ** 0.5)
        self.W_enc.data = w
        self.W_dec.data = w.T.contiguous() / w.T.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        top_v, top_i = pre.topk(self.k, dim=-1)
        z = torch.zeros_like(pre)
        z.scatter_(-1, top_i, F.relu(top_v))
        return z, top_i, top_v
    def decode(self, z):
        return z @ self.W_dec + self.b_dec
    def forward(self, x):
        z, _, _ = self.encode(x)
        return self.decode(z), z

def train_sae(X, d_in, n, k, epochs, bs, lr, name=''):
    X = X.cuda()
    sae = TopKSAE(d_in, n, k).cuda()
    opt = torch.optim.Adam(sae.parameters(), lr=lr)
    N = X.shape[0]
    last_fire = torch.zeros(n, device='cuda')
    step = 0
    for e in range(epochs):
        perm = torch.randperm(N, device='cuda')
        el = 0.0; nb = 0
        for i in range(0, N, bs):
            xb = X[perm[i:i+bs]]
            xh, z = sae(xb)
            loss = F.mse_loss(xh, xb)
            opt.zero_grad(); loss.backward(); opt.step()
            with torch.no_grad():
                sae.W_dec.data /= sae.W_dec.data.norm(dim=-1, keepdim=True).clamp_min(1e-8)
                fired = (z > 0).any(dim=0)
                last_fire[fired] = step
            step += 1
            el += loss.item(); nb += 1
        if (e+1) % 5 == 0 or e == 0:
            dead = (step - last_fire) > 500
            n_dead = int(dead.sum().item())
            if n_dead > 0:
                with torch.no_grad():
                    new_dirs = torch.randn(n_dead, d_in, device='cuda')
                    new_dirs /= new_dirs.norm(dim=-1, keepdim=True)
                    sae.W_dec.data[dead] = new_dirs
                    sae.W_enc.data[:, dead] = new_dirs.T
                    sae.b_enc.data[dead] = 0.0
            with torch.no_grad():
                xh, _ = sae(X[:10000])
                var_expl = 1 - ((xh - X[:10000]).var() / X[:10000].var()).item()
            print(f'  {name} ep{e+1:>2}/{epochs} mse={el/nb:.5f} var_expl={var_expl:.3f} dead_revived={n_dead}')
    return sae.cpu()

saes_moe = {}
for L in CAPTURE_LAYERS:
    print(f'\n=== Training MoE SAE on L{L} (n={N_FEATURES}, k={TOPK}) ===')
    sae = train_sae(acts_moe[L], D_MODEL, N_FEATURES, TOPK,
                    SAE_EPOCHS, SAE_BS, SAE_LR, name=f'L{L}_moe')
    saes_moe[L] = sae
    torch.save(sae.state_dict(), OUT / f'sae_moe_L{L}_n{N_FEATURES}_k{TOPK}.pt')
    torch.cuda.empty_cache()

## Cell 7 — Encode + compute healthy masks

In [ ]:
feature_fires = {}
feature_pre = {}
for L in CAPTURE_LAYERS:
    sae = saes_moe[L].cuda()
    X = acts_moe[L].cuda()
    N = X.shape[0]
    fires = torch.zeros(N, N_FEATURES, dtype=torch.bool)
    pre = torch.zeros(N, N_FEATURES, dtype=torch.float16)
    bs = 8192
    b_dec = sae.b_dec; W_enc = sae.W_enc; b_enc = sae.b_enc
    with torch.no_grad():
        for i in tqdm(range(0, N, bs), desc=f'encode L{L}'):
            xb = X[i:i+bs]
            z, _, _ = sae.encode(xb)
            p = (xb - b_dec) @ W_enc + b_enc
            fires[i:i+bs] = (z > 0).cpu()
            pre[i:i+bs] = p.to(torch.float16).cpu()
    feature_fires[L] = fires
    feature_pre[L] = pre
    sae.cpu(); torch.cuda.empty_cache()
    fr = fires.float().mean(dim=0)
    healthy = ((fr > 0.005) & (fr < 0.30)).sum().item()
    print(f'L{L} MoE: healthy={healthy}/{N_FEATURES}  mean_fire={fr.mean()*100:.2f}%')

healthy_mask = {}
for L in CAPTURE_LAYERS:
    fr = feature_fires[L].float().mean(dim=0)
    healthy_mask[L] = (fr > 0.005) & (fr < 0.30)

## Cell 8 — Critical test: fire→pre correlation across MoE layers

If hypothesis B holds, this should show `max |corr| > 0.1` (vs 0.009 on residual).

In [ ]:
pairs = [(11, 17), (17, 23), (11, 23)]

Tm = 100
summary = {}
for a, b in pairs:
    fires_a = feature_fires[a].float().cuda()
    pre_b = feature_pre[b].float().cuda()
    N = fires_a.shape[0]
    fa = fires_a - fires_a.mean(0); fa = fa / fa.std(0).clamp_min(1e-6)
    pb = pre_b - pre_b.mean(0);     pb = pb / pb.std(0).clamp_min(1e-6)
    corr = (fa.T @ pb) / N
    
    supp_a = feature_fires[a].float().sum(0)
    supp_b = feature_fires[b].float().sum(0)
    mr = healthy_mask[a] & (supp_a >= Tm)
    mc = healthy_mask[b] & (supp_b >= Tm)
    corr_q = corr[mr][:, mc].cpu()
    
    summary[f'L{a}->L{b}'] = dict(
        max_abs_corr=float(corr_q.abs().max().item()),
        frac_gt_01=float((corr_q.abs()>0.1).float().mean().item()*100),
        frac_gt_02=float((corr_q.abs()>0.2).float().mean().item()*100))
    
    print(f'\n=== MoE L{a}->L{b} fires->pre corr {tuple(corr_q.shape)} ===')
    print(f'  abs>0.1: {summary[f"L{a}->L{b}"]["frac_gt_01"]:.3f}%')
    print(f'  abs>0.2: {summary[f"L{a}->L{b}"]["frac_gt_02"]:.3f}%')
    print(f'  max |corr|: {summary[f"L{a}->L{b}"]["max_abs_corr"]:.4f}')
    
    rows_idx = mr.nonzero().flatten()
    cols_idx = mc.nonzero().flatten()
    vals, idxs = corr_q.abs().view(-1).topk(10)
    n_cols = corr_q.shape[1]
    print('  top-10 edges by |corr|:')
    for v, fi in zip(vals.tolist(), idxs.tolist()):
        i = fi // n_cols; j = fi % n_cols
        src = rows_idx[i].item(); dst = cols_idx[j].item()
        print(f'    f{src:>4}->f{dst:>4}  corr={corr_q[i,j].item():+.4f}')
    del fires_a, pre_b, fa, pb, corr; torch.cuda.empty_cache()

# Headline comparison
print('\n' + '='*60)
print('HEADLINE: MoE-level vs residual-level max |corr|')
print('residual (Phase A): 0.009')
for k, v in summary.items():
    print(f'  MoE {k}: {v["max_abs_corr"]:.4f}')

## Cell 9 — Upload

In [ ]:
create_repo(HF_OUT, repo_type='model', private=False, exist_ok=True)
api = HfApi()

np.savez(OUT / 'moe_features.npz',
         **{f'L{L}_fires': feature_fires[L].numpy() for L in CAPTURE_LAYERS},
         **{f'L{L}_healthy': healthy_mask[L].numpy() for L in CAPTURE_LAYERS})

with open(OUT / 'summary_corr.json', 'w') as f:
    json.dump(summary, f, indent=2)

readme = f'''---
license: apache-2.0
base_model: Qwen/Qwen3.6-35B-A3B
tags: [sparse-autoencoder, feature-circuits, mechanistic-interpretability, hybrid-moe-gdn]
---

# Qwen3.6-35B-A3B Feature Circuits — Phase B (MoE sub-block)

Extension of Phase A. Phase A tested residual stream and found null (max corr 0.009 across layers).
Phase B tests MoE sub-block outputs directly.

## Files (phase_b/)

- `sae_moe_L{{11,17,23}}_n{N_FEATURES}_k{TOPK}.pt` — MoE-output TopK SAEs
- `moe_features.npz` — feature fires + healthy masks
- `summary_corr.json` — headline correlation numbers

## Results

See `summary_corr.json`. Compare to Phase A residual null (max |corr|=0.009).
'''
with open(OUT / 'README_phase_b.md', 'w') as f: f.write(readme)

for fp in list(OUT.glob('sae_moe_*.pt')) + [OUT/'moe_features.npz',
                                               OUT/'summary_corr.json',
                                               OUT/'README_phase_b.md']:
    try:
        api.upload_file(path_or_fileobj=str(fp), path_in_repo=f'phase_b/{fp.name}',
                        repo_id=HF_OUT, repo_type='model',
                        commit_message='FC Phase B: ' + fp.name)
        print(f'OK {fp.name}')
    except Exception as e:
        print(f'FAIL {fp.name}: {e}')
print(f'\nOK https://huggingface.co/{HF_OUT}/tree/main/phase_b')